# Los datos centrales

La traza de la red tiene geometría pero le falta fechas,
las estaciones tienen fechas,
y los censos estan por separados a los datos ferroviarios. Este pipeline procesa cada cosa y
deja tres tablas, vías, estaciones y gente, de donde salen todos los análisis del 2_ y todos los
datos de la web app del 3_.

## Datos

- `Ferrocarril.geojson` — `geometry` de la traza, `FUN` como única fuente del estado actual y `kms`,
  el largo declarado de cada tramo.
- `osm_rail_abandoned.json` — `geometry` de la vía levantada o sin servicio que el IDE no tiene.
- `osm_rutas_tren.json` — `tags.service`, `tags.public_transport:version` y la `geometry` de los
  ways de cada ruta, para saber qué vía recorre hoy un servicio de pasajeros.
- `Estacion_Ferroviaria.geojson` — `geometry` y `nombre` de las estaciones del IDE.
- `wikidata_estaciones.csv` — `st` para deduplicar, `stLabel`, `coord`, `opened` e `inception` como
  año de alta y `closed` como año de baja.
- `radios{censo}.csv` — `wkt`, `codigo`, `cod_prov` y la población de cada radio de los cuatro censos.
- `ondat_ffcc_hist.xls` — año y km de red del ONDaT, el único control externo.
- `ign_provincia.json` — sólo `geometry`, de mapa base.

In [11]:
import json, os
import numpy as np, pandas as pd, geopandas as gpd
from scipy.spatial import cKDTree
from shapely import from_wkt
from shapely.geometry import LineString, box
from shapely.strtree import STRtree
from pathlib import Path

RAW = Path('data/raw')
SALIDA = Path('data/pipeline')
SALIDA.mkdir(parents=True, exist_ok=True)

CRS_METRICO = 5346           # POSGAR 2007/3: toda distancia se mide con esto
CRS_AREA = 'ESRI:102033'     # área igual, para el área de los radios
DUPLICADO = 250              # metros: vía de OSM más cerca que esto del IDE es la misma vía
SIMPLIFICACION = 0.002       # grados (~200 m) de tolerancia al simplificar la vía
TOLERANCIA_PASAJEROS = 300   # metros entre una vía y el recorrido de pasajeros que la usa
MISMA_ESTACION = 1000        # metros: una de Wikidata más cerca que esto de una del IDE es la misma
ESTACION_SOBRE_VIA = 1000    # metros: una estación más lejos que esto de toda vía no cuenta
ESTACION_EN_VIA = 500        # metros entre una estación y la vía de la que toma la categoría
UMBRAL_KM = 10               # km: tener una estación a menos de esto es tenerla cerca
ANIO_HOY = 2025

# censo: columna de población; la de 2022 es sólo la gente en viviendas particulares
CENSOS = {1991: 'B_POB_TOT', 2001: 'POB_TOT', 2010: 'B_POB_TOT', 2022: 'POB_TOT_P'}
PROVINCIAS = {'02': 'CABA', '06': 'Buenos Aires', '10': 'Catamarca', '14': 'Córdoba',
              '18': 'Corrientes', '22': 'Chaco', '26': 'Chubut', '30': 'Entre Ríos',
              '34': 'Formosa', '38': 'Jujuy', '42': 'La Pampa', '46': 'La Rioja',
              '50': 'Mendoza', '54': 'Misiones', '58': 'Neuquén', '62': 'Río Negro',
              '66': 'Salta', '70': 'San Juan', '74': 'San Luis', '78': 'Santa Cruz',
              '82': 'Santa Fe', '86': 'Sgo. del Estero', '90': 'Tucumán', '94': 'Tierra del Fuego'}
PASAJEROS, CARGA = 'con tren de pasajeros', 'sólo tren de carga'
CERRADA, NUNCA = 'perdió el tren', 'nunca tuvo tren'

## Las estaciones de Wikidata

Wikidata es la única fuente de fechas: de cada estación tomo el año de `opened`, o el de
`inception` si falta, como alta, y el de `closed` como baja. Deduplico por `st`. Las que no tienen
coordenadas quedan igual, porque sirven para contar cierres por año aunque no se puedan ubicar.

In [12]:
wd = pd.read_csv(RAW / 'wikidata_estaciones.csv').drop_duplicates('st').reset_index(drop=True)

def anio_de(col):
    return pd.to_datetime(wd[col], errors='coerce', utc=True).dt.year

# coord viene como el texto "Point(lon lat)".
lon = wd.coord.str.extract(r'Point\(([-\d.]+)')[0].astype(float)
lat = wd.coord.str.extract(r'Point\([-\d.]+ ([-\d.]+)')[0].astype(float)
puntos_wd = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs=4326).to_crs(CRS_METRICO)
# Sin coordenadas, la proyección devuelve infinito: queda vacío.
wd = pd.DataFrame({'fuente': 'Wikidata', 'nombre': wd.stLabel,
                   'x': puntos_wd.x.where(lon.notna()), 'y': puntos_wd.y.where(lon.notna()),
                   'alta': anio_de('opened').fillna(anio_de('inception')), 'baja': anio_de('closed')})
con_coordenadas = wd.x.notna()
print(f'{len(wd)} estaciones · {con_coordenadas.sum()} con coordenadas · '
      f'{(con_coordenadas & wd.alta.notna()).sum()} con año de alta · {(con_coordenadas & wd.baja.notna()).sum()} con año de baja')

3060 estaciones · 2766 con coordenadas · 1706 con año de alta · 691 con año de baja


## La traza, partida en pedazos dibujables

Exploto los 318 MultiLineString del IDE en partes, porque una fila sola puede medir 838 km y un
cambio de estado apagaría 300 km de golpe. Los `kms` declarados de cada tramo se reparten entre sus
partes según el largo medido, así los totales dan las cifras oficiales. A eso le sumo la vía de OSM
a más de 250 m del IDE, que entra como no operativa y con el largo medido, porque OSM no declara
kilometraje. Ese `km` es el único largo de vía de todo el proyecto.

In [13]:
ide = gpd.read_file(RAW / 'Ferrocarril.geojson')
ide['tramo'] = ide.index
ide['activo'] = ide.FUN.str.lower().str.startswith(('activo', 'operativ'))
ide = ide.explode(index_parts=False).reset_index(drop=True)
largo = ide.to_crs(CRS_METRICO).length / 1000
ide['km'] = ide.kms * largo / largo.groupby(ide.tramo).transform('sum')
ide['fuente'] = 'IDE'

ways = json.load(open(RAW / 'osm_rail_abandoned.json'))['elements']
lineas = []
for w in ways:
    if len(w.get('geometry', [])) > 1:
        lineas.append(LineString([(q['lon'], q['lat']) for q in w['geometry']]))
osm = gpd.GeoDataFrame(geometry=lineas, crs=4326)
# La vía de OSM pegada a una del IDE es la misma vía mapeada dos veces.
lejos = osm.to_crs(CRS_METRICO).distance(ide.to_crs(CRS_METRICO).union_all()) > DUPLICADO
osm = osm[lejos].reset_index(drop=True)
osm['activo'] = False
osm['km'] = osm.to_crs(CRS_METRICO).length / 1000
osm['fuente'] = 'OSM'

COLUMNAS = ['fuente', 'activo', 'km', 'geometry']
vias = gpd.GeoDataFrame(pd.concat([ide[COLUMNAS], osm[COLUMNAS]], ignore_index=True), crs=4326)
print(vias.groupby('fuente').km.agg(partes='size', km='sum').round().to_string())

        partes       km
fuente                 
IDE       4203  36055.0
OSM       3729  10283.0


## Las fechas de la vía

Cada parte toma el año de alta de la estación de Wikidata con alta más cercana, y el de baja de la
más cercana con baja. La distancia hasta esa estación queda como medida de confianza de cada fecha.
La baja sólo tiene sentido en la vía que hoy no lleva trenes; en la activa es el año en que perdió
los pasajeros, si hoy sólo lleva carga.

In [14]:
metrica = vias.to_crs(CRS_METRICO).geometry.values

def fecha_mas_cercana(col):
    """El año `col` de la estación fechada más cercana a cada parte, y la distancia en km."""
    fechadas = wd[con_coordenadas & wd[col].notna()]
    puntos = gpd.GeoSeries(gpd.points_from_xy(fechadas.x, fechadas.y), crs=CRS_METRICO)
    cercana = puntos.sindex.nearest(metrica, return_all=False)[1]
    distancia = puntos.values[cercana].distance(metrica) / 1000
    return fechadas[col].values[cercana], distancia

vias['alta'], vias['dist_alta_km'] = fecha_mas_cercana('alta')
vias['baja'], vias['dist_baja_km'] = fecha_mas_cercana('baja')
print(f'distancia a la estación de alta: mediana {vias.dist_alta_km.median():.1f} km · '
      f'de baja, en la vía no operativa: {vias.dist_baja_km[~vias.activo].median():.1f} km')

distancia a la estación de alta: mediana 5.4 km · de baja, en la vía no operativa: 24.6 km


### Otras formas de fechar la vía

El fechado también se puede plantear como aprendizaje supervisado, suponiendo que estaciones y
partes de vía son intercambiables: un modelo aprende el año de las estaciones a partir de su
posición y lo predice para cada parte. Probé k vecinos, una red neuronal, gradient boosting, un
proceso gaussiano y una mezcla que toma el alta de un modelo y la baja de otro. Cada modelo elige su
configuración por validación cruzada sobre las estaciones, y el panel da ese error en años para el
alta y la baja. El ONDaT no elige: sólo mide la curva, con su error cuadrático medio y la
correlación entre sus cambios a 5 años y los de la red real.

![Los km de cada método de fechado contra el ONDaT](data/pipeline/comparacion_metodos_fechado.png)

En el archivo 1bis_otras_formas_de_fechar.ipynb dejo ese analisis

## La categoría de hoy

Simplifico la geometría a unos 200 m, que a escala nacional no se ven, y desde acá todo se mide
sobre ella. Una parte activa lleva pasajeros si la mitad de su largo cae a menos de 300 m de la vía
que recorre una ruta de OpenStreetMap que sea un servicio de pasajeros no turístico; si no, es de
carga. La no operativa es inactiva si está en el IDE y levantada si sólo está en OSM.

In [15]:
vias['geometry'] = vias.geometry.simplify(SIMPLIFICACION)
vias = vias.to_crs(CRS_METRICO)

# Una ruta es un servicio de pasajeros si lo dice su etiqueta, que puede traer varios tipos separados
# por punto y coma, o si sigue el esquema de transporte público sin decir el tipo.
SERVICIOS = {'commuter', 'regional', 'long_distance', 'national'}
recorridos = []
for e in json.load(open(RAW / 'osm_rutas_tren.json'))['elements']:
    t = e.get('tags', {})
    tipos = set(t.get('service', '').split(';'))
    if not (tipos & SERVICIOS or ('service' not in t and 'public_transport:version' in t)):
        continue
    for m in e['members']:
        if m['type'] == 'way' and len(m.get('geometry', [])) >= 2:
            recorridos.append(LineString([(p['lon'], p['lat']) for p in m['geometry']]))
banda = gpd.GeoSeries(recorridos, crs=4326).to_crs(CRS_METRICO).buffer(TOLERANCIA_PASAJEROS).union_all()
pasa = vias.intersection(banda).length / vias.length >= .5

vias['categoria'] = np.select([vias.activo & pasa, vias.activo, vias.fuente == 'IDE'],
                              ['pasajeros', 'carga', 'inactiva'], default='levantada')
print(vias.groupby('categoria').km.sum().round().astype(int).to_string())

categoria
carga        17417
inactiva     14006
levantada    10283
pasajeros     4632


## Las estaciones que cuentan

Las estaciones son las del IDE más las de Wikidata que no tienen una del IDE a menos de 1 km. Cada estación que cuenta queda unida a las
partes que tiene a menos de 500 m, que son las que le dan su categoría año por año.

In [16]:
puntos_ide = gpd.read_file(RAW / 'Estacion_Ferroviaria.geojson').to_crs(CRS_METRICO)
# Las estaciones del IDE vienen como MultiPoint: representative_point las vuelve un punto.
puntos_ide['geometry'] = puntos_ide.geometry.representative_point()
ide_est = pd.DataFrame({'fuente': 'IDE', 'nombre': puntos_ide.nombre,
                        'x': puntos_ide.geometry.x, 'y': puntos_ide.geometry.y})
estaciones = pd.concat([ide_est, wd], ignore_index=True)

ubicada = estaciones.x.notna().values
xy = estaciones[['x', 'y']].values
repetida = np.zeros(len(estaciones), bool)
de_wikidata = (estaciones.fuente == 'Wikidata').values & ubicada
repetida[de_wikidata] = cKDTree(ide_est[['x', 'y']].values).query(xy[de_wikidata])[0] <= MISMA_ESTACION
puntos = gpd.GeoSeries(gpd.points_from_xy(estaciones.x, estaciones.y), crs=CRS_METRICO)
distancia_a_via = np.full(len(estaciones), np.inf)
distancia_a_via[ubicada] = vias.sindex.nearest(puntos[ubicada], return_all=False, return_distance=True)[1]
estaciones['cuenta'] = ubicada & ~repetida & (distancia_a_via <= ESTACION_SOBRE_VIA)

cuentan = np.flatnonzero(estaciones.cuenta)
pares = STRtree(vias.geometry.values).query(puntos.values[cuentan], predicate='dwithin', distance=ESTACION_EN_VIA)
estaciones_vias = pd.DataFrame({'estacion': cuentan[pares[0]], 'parte': pares[1]})
print(estaciones[estaciones.cuenta].fuente.value_counts().to_string())
print(f'{len(estaciones_vias)} pares estación–parte a menos de {ESTACION_EN_VIA} m')

fuente
IDE         2217
Wikidata     844
6970 pares estación–parte a menos de 500 m


## La categoría de un año

Una parte no existe antes de su alta. Lo que hoy lleva pasajeros los llevó desde el alta, lo que hoy
es de carga los perdió en su baja, y lo que hoy no funciona llevó pasajeros hasta su baja y después
queda inactivo o levantado, porque no hay fecha de levantamiento. Una estación toma la mejor
categoría de sus partes: con pasajeros, de carga o cerrada. Cada radio censal cae en una de cuatro
según las estaciones que tiene a menos de 10 km: con tren de pasajeros, sólo de carga, perdió el
tren si sólo hay estaciones cerradas, y nunca tuvo tren si no hay ninguna.

In [17]:
ALTA = vias.alta.values
FIN_PASAJEROS = np.where(vias.categoria == 'pasajeros', np.inf, vias.baja)
FIN_SERVICIO = np.where(vias.categoria.isin(['pasajeros', 'carga']), np.inf, vias.baja)
CATEGORIA_HOY = vias.categoria.values
PAR_ESTACION = estaciones_vias.estacion.values
PAR_PARTE = estaciones_vias.parte.values

def categoria_en(anio):
    """La categoría de cada parte en un año, o '' si todavía no estaba construida."""
    construida = ALTA <= anio
    pasajeros = construida & (anio < FIN_PASAJEROS)
    carga = construida & ~pasajeros & (anio < FIN_SERVICIO)
    cerrada = construida & ~pasajeros & ~carga
    return np.select([pasajeros, carga, cerrada], ['pasajeros', 'carga', CATEGORIA_HOY], default='')

def estaciones_sobre(cat, categorias):
    """Las estaciones que tienen a menos de ESTACION_EN_VIA alguna parte de esas categorías."""
    marca = np.zeros(len(estaciones), bool)
    np.logical_or.at(marca, PAR_ESTACION, np.isin(cat[PAR_PARTE], categorias))
    return marca

def hay_estacion(xy, mascara):
    """Si cada punto tiene a menos de UMBRAL_KM alguna de las estaciones marcadas."""
    if not mascara.any():
        return np.zeros(len(xy), bool)
    d = cKDTree(estaciones.loc[mascara, ['x', 'y']].values).query(xy, distance_upper_bound=UMBRAL_KM * 1000)[0]
    return d < np.inf

def acceso(xy, anio):
    cat = categoria_en(anio)
    return np.select([hay_estacion(xy, estaciones_sobre(cat, ['pasajeros'])),
                      hay_estacion(xy, estaciones_sobre(cat, ['carga'])),
                      hay_estacion(xy, estaciones_sobre(cat, ['pasajeros', 'carga', 'inactiva', 'levantada']))],
                     [PASAJEROS, CARGA, CERRADA], default=NUNCA)

print('la categoría de', ANIO_HOY, 'es la de hoy:', (categoria_en(ANIO_HOY) == CATEGORIA_HOY).all())

la categoría de 2025 es la de hoy: True


## La gente

Cada radio censal se reduce a un punto, que es lo que hace falta para medir distancias, y guarda el
área de su polígono para las densidades. Los radios cambian de forma y de código entre censos, así
que cada uno va al departamento del radio de 2022 más cercano. Cada radio lleva dos categorías: la
de la red del año de su censo y la de la red de hoy.

In [18]:
radios = []
for censo, columna in CENSOS.items():
    d = pd.read_csv(RAW / f'radios{censo}.csv')
    d.columns = [col.strip().lstrip('﻿') for col in d.columns]   # el CSV trae BOM
    poligonos = gpd.GeoSeries(from_wkt(d['wkt']), crs=4326)
    punto = poligonos.representative_point()
    r = pd.DataFrame({'censo': censo,
                      'codigo': d['codigo'].astype(str).str.zfill(9),
                      'cod_prov': d['cod_prov'].astype(str).str.zfill(2),
                      'pob': pd.to_numeric(d[columna], errors='coerce').fillna(0),
                      'area_km2': poligonos.to_crs(CRS_AREA).area.values / 1e6,
                      'lon': punto.x.values, 'lat': punto.y.values})
    # Unos pocos radios caen fuera del país por coordenadas mal cargadas.
    r = r[(r.lat < -20) & (r.lat > -56) & (r.lon < -52) & (r.lon > -74)]
    metrico = gpd.GeoSeries(gpd.points_from_xy(r.lon, r.lat), crs=4326).to_crs(CRS_METRICO)
    r['x'] = metrico.x.values
    r['y'] = metrico.y.values
    radios.append(r.drop(columns=['lon', 'lat']))
radios = pd.concat(radios, ignore_index=True)

r22 = radios[radios.censo == 2022]
cercano = cKDTree(r22[['x', 'y']].values).query(radios[['x', 'y']].values)[1]
radios['dep'] = r22.codigo.str[:5].values[cercano]

xy = radios[['x', 'y']].values
radios['acceso'] = ''
for censo in CENSOS:
    es = (radios.censo == censo).values
    radios.loc[es, 'acceso'] = acceso(xy[es], censo)
radios['acceso_hoy'] = acceso(xy, ANIO_HOY)
millones = radios.pivot_table(index='censo', columns='acceso', values='pob', aggfunc='sum') / 1e6
print(millones[[PASAJEROS, CARGA, CERRADA, NUNCA]].round(2).to_string())

acceso  con tren de pasajeros  sólo tren de carga  perdió el tren  nunca tuvo tren
censo                                                                             
1991                    24.70                2.35            2.16             3.40
2001                    22.10                5.69            4.64             3.83
2010                    23.29                7.20            5.44             4.19
2022                    24.71                9.32            6.90             4.68


## El mapa base y el ONDaT

Las provincias del IGN traen la costa al detalle, que a escala nacional no se ve: las simplifico a
0,01° (~1 km) y les saco el antártico, que llega a latitud -90. Del ONDaT tomo el año y los km de red, que empiezan en la fila 14 de su planilla.

In [19]:
# Las provincias vienen en un solo feature gigante y OGR lo rechaza por tamaño.
os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'
provincias = gpd.read_file(RAW / 'ign_provincia.json')[['geometry']]
provincias['geometry'] = provincias.geometry.simplify(0.01)
provincias = provincias.clip(box(-74, -56, -53, -21)).to_crs(CRS_METRICO)

ondat = pd.read_excel(RAW / 'ondat_ffcc_hist.xls', header=None).iloc[13:172, 0:2]
ondat.columns = ['anio', 'red_km']
ondat = ondat.apply(pd.to_numeric, errors='coerce').dropna()
ondat['anio'] = ondat.anio.astype(int)
print(f'{len(provincias)} polígonos de provincia · ONDaT {ondat.anio.min()}–{ondat.anio.max()}, {len(ondat)} años con dato')

24 polígonos de provincia · ONDaT 1857–2000, 142 años con dato


## La salida

Todo va a `data/pipeline`, con las coordenadas en EPSG:5346.

- `vias.parquet` — una fila por parte: `fuente`, `activo`, `km`, `alta`, `baja`, `dist_alta_km`,
  `dist_baja_km`, `categoria` (la de hoy) y `geometry`.
- `estaciones.parquet` — una fila por estación del IDE o de Wikidata: `fuente`, `nombre`, `x`, `y`
  (vacías si no tiene coordenadas), `alta` y `baja` (sólo Wikidata) y `cuenta`.
- `estaciones_vias.csv` — `estacion` y `parte`, las filas de las dos tablas de arriba a menos de
  500 m.
- `radios.parquet` — una fila por radio de cada censo: `censo`, `codigo`, `cod_prov`, `dep`, `pob`,
  `area_km2`, `x`, `y`, `acceso` (con la red de su censo) y `acceso_hoy`.
- `provincias.parquet` y `provincias_nombres.csv` — el mapa base y el nombre de cada `cod_prov`.
- `ondat.csv` — `anio` y `red_km`.

In [20]:
vias.to_parquet(SALIDA / 'vias.parquet')
estaciones.to_parquet(SALIDA / 'estaciones.parquet', index=False)
estaciones_vias.to_csv(SALIDA / 'estaciones_vias.csv', index=False)
radios.to_parquet(SALIDA / 'radios.parquet', index=False)
provincias.to_parquet(SALIDA / 'provincias.parquet')
pd.Series(PROVINCIAS, name='nombre').rename_axis('cod_prov').to_csv(SALIDA / 'provincias_nombres.csv')
ondat.to_csv(SALIDA / 'ondat.csv', index=False)
print(f'{len(vias)} partes · {estaciones.cuenta.sum()} estaciones que cuentan · {len(radios)} radios')

7932 partes · 3061 estaciones que cuentan · 205583 radios
